<a href="https://colab.research.google.com/github/kraszor/SIGK-2025Z/blob/main/transformacja-3d/sigk4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install trimesh

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import trimesh
import pandas as pd
from scipy.spatial.distance import cdist

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Używam urządzenia: {device}")
if device.type == 'cuda':
    print(f"   Karta graficzna: {torch.cuda.get_device_name(0)}")

🖥️ Używam urządzenia: cuda
   Karta graficzna: Tesla T4


In [17]:
class Canonicalizer:
    """Class responsible for normalization of the object."""
    def __init__(self):
        self.centroid = None
        self.scale = None

    def fit_transform(self, mesh):
        vertices = mesh.vertices
        self.centroid = np.mean(vertices, axis=0)
        vertices = vertices - self.centroid
        self.scale = np.max(np.linalg.norm(vertices, axis=1))
        vertices = vertices / self.scale
        mesh.vertices = vertices
        return mesh

    def transform(self, mesh):
        vertices = mesh.vertices
        centroid = np.mean(vertices, axis=0)
        vertices = vertices - centroid
        scale = np.max(np.linalg.norm(vertices, axis=1))
        return vertices / scale


class PositionalEncoding(nn.Module):
    def __init__(self, num_freqs=6):
        super().__init__()
        self.num_freqs = num_freqs
        freq_bands = 2.0 ** torch.linspace(0.0, num_freqs - 1, num_freqs)
        self.register_buffer('freq_bands', freq_bands)

    def forward(self, x):
        embed_fns = []
        embed_fns.append(x)

        for freq in self.freq_bands:
            embed_fns.append(torch.sin(x * freq * np.pi))
            embed_fns.append(torch.cos(x * freq * np.pi))

        return torch.cat(embed_fns, dim=-1)

class DisplacementField(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_freqs = 3
        self.embedder = PositionalEncoding(num_freqs=self.num_freqs)

        input_dim = 3 + (3 * 2 * self.num_freqs)
        hidden_dim = 256
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)
        self.layer4 = nn.Linear(hidden_dim + input_dim, hidden_dim)
        self.layer5 = nn.Linear(hidden_dim, hidden_dim)
        self.layer6 = nn.Linear(hidden_dim, hidden_dim)
        self.out_layer = nn.Linear(hidden_dim, 3)

        self.act = nn.ReLU()
        self.out_act = nn.Tanh()

    def forward(self, x):
        x_emb = self.embedder(x)
        h = self.act(self.layer1(x_emb))
        h = self.act(self.layer2(h))
        h = self.act(self.layer3(h))

        h = torch.cat([h, x_emb], dim=-1)

        h = self.act(self.layer4(h))
        h = self.act(self.layer5(h))
        h = self.act(self.layer6(h))

        # Wyjście
        return self.out_act(self.out_layer(h))


In [4]:
def chamfer_distance(p1, p2):
    """
    Calculates the Chamfer distance between two sets of points.
    p1: (N, 3), p2: (M, 3)
    """
    x = p1.unsqueeze(1)
    y = p2.unsqueeze(0)

    dist = torch.norm(x - y, dim=2)

    min_dist_p1, _ = torch.min(dist, dim=1)
    min_dist_p2, _ = torch.min(dist, dim=0)

    return torch.mean(min_dist_p1) + torch.mean(min_dist_p2)

In [20]:
def train_deformation(source_name, source_path, target_path, epochs=1000, lr=0.001):
    print(f"\n🚀 Rozpoczynam trening PRO: {source_name} -> Teapot")

    try:
        s_mesh = trimesh.load(source_path)
        t_mesh = trimesh.load(target_path)
    except Exception as e:
        print(f"❌ Błąd: {e}")
        return None, None

    canon = Canonicalizer()
    s_mesh = canon.fit_transform(s_mesh)
    t_mesh = Canonicalizer().fit_transform(t_mesh)

    t_points_np, _ = trimesh.sample.sample_surface(t_mesh, 10000)
    t_points = torch.tensor(t_points_np, dtype=torch.float32).to(device)

    model = DisplacementField().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=50)

    model.train()

    for i in range(epochs):
        optimizer.zero_grad()

        s_points_np, _ = trimesh.sample.sample_surface(s_mesh, 10000)
        s_points = torch.tensor(s_points_np, dtype=torch.float32).to(device)

        displacement = model(s_points)
        deformed_points = s_points + displacement

        chamfer = chamfer_distance(deformed_points, t_points)

        smoothness_loss = torch.mean(displacement ** 2)

        loss = chamfer + (0.15 * smoothness_loss)

        loss.backward()
        optimizer.step()

        scheduler.step(loss)

        if i % 100 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"   Epoka {i}/{epochs} | Loss: {loss.item():.6f} | LR: {current_lr:.6f}")

    print("✅ Trening zakończony.")
    return model, s_mesh, t_mesh

In [15]:
from scipy.spatial import cKDTree

def predict_in_batches(model, vertices, batch_size=10000):
    """
    Przepuszcza wierzchołki przez sieć w małych paczkach,
    aby nie zapchać VRAM/RAM.
    """
    model.eval()
    n_verts = len(vertices)
    displacements = []

    with torch.no_grad():
        for i in range(0, n_verts, batch_size):
            batch_cpu = vertices[i : i + batch_size]
            batch_gpu = torch.tensor(batch_cpu, dtype=torch.float32).to(device)

            disp_gpu = model(batch_gpu)

            displacements.append(disp_gpu.cpu().numpy())

    return np.vstack(displacements)

def calculate_metrics_optimized(source_deformed, target_mesh, num_samples=5000):
    """
    Wersja super-lekka. Upraszcza siatkę przed ray-tracingiem.
    """
    p_pred, _ = trimesh.sample.sample_surface(source_deformed, 2048)
    p_target, _ = trimesh.sample.sample_surface(target_mesh, 2048)

    tree_target = cKDTree(p_target)
    dist_pred_to_target, _ = tree_target.query(p_pred, k=1)

    tree_pred = cKDTree(p_pred)
    dist_target_to_pred, _ = tree_pred.query(p_target, k=1)

    chamfer = np.mean(dist_pred_to_target) + np.mean(dist_target_to_pred)

    try:
        low_poly_pred = source_deformed.copy()
        low_poly_target = target_mesh.copy()

        try:
            low_poly_pred = low_poly_pred.simplify_quadratic_decimation(5000)
            low_poly_target = low_poly_target.simplify_quadratic_decimation(5000)
        except Exception:
            pass

        bbox_min = np.min([low_poly_pred.bounds[0], low_poly_target.bounds[0]], axis=0)
        bbox_max = np.max([low_poly_pred.bounds[1], low_poly_target.bounds[1]], axis=0)

        test_points = np.random.uniform(bbox_min, bbox_max, (5000, 3))

        contains_pred = low_poly_pred.contains(test_points)
        contains_target = low_poly_target.contains(test_points)

        intersection = np.sum(contains_pred & contains_target)
        union = np.sum(contains_pred | contains_target)

        jaccard = intersection / union if union > 0 else 0.0
        dice = 2 * intersection / (np.sum(contains_pred) + np.sum(contains_target)) if union > 0 else 0.0

        del low_poly_pred
        del low_poly_target
        del test_points

    except Exception as e:
        print(f"⚠️ Pominięto Jaccard/Dice z powodu błędu: {e}")
        jaccard, dice = 0.0, 0.0

    return chamfer, jaccard, dice

def save_and_evaluate(model, source_mesh, target_mesh, name, output_dir="./drive/MyDrive/sigk_4/results"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"📊 Generowanie wyników dla: {name}...")
    dense_source = source_mesh.copy()

    dense_source = dense_source.subdivide()

    verts_cpu = dense_source.vertices
    final_displacement = predict_in_batches(model, verts_cpu, batch_size=5000)

    steps = [0.0, 0.5, 1.0]
    final_deformed_mesh = None

    for t in steps:
        new_verts = verts_cpu + (final_displacement * t)
        mesh_copy = source_mesh.copy()
        mesh_copy.vertices = new_verts

        filename = f"{output_dir}/{name}_step_{int(t*100)}.obj"
        mesh_copy.export(filename)
        print(f"Zapisano {filename}")

        if t == 1.0:
            final_deformed_mesh = mesh_copy

    # chamfer, jaccard, dice = calculate_metrics_optimized(final_deformed_mesh, target_mesh)

    # return {
    #     "Object": name,
    #     "Chamfer": chamfer,
    #     "Jaccard (IoU)": jaccard,
    #     "Dice": dice
    # }

In [21]:
import gc
# del s_mesh
# del t_mesh

DATA_DIR = "./drive/MyDrive/sigk_4/data"
TARGET_FILE = os.path.join(DATA_DIR, "teapot.obj")

OBJECTS = {
"Bunny": os.path.join(DATA_DIR, "bunny.obj"),
"Dragon": os.path.join(DATA_DIR, "dragon.obj"),
"Armadillo": os.path.join(DATA_DIR, "armadillo.obj")
}

ASIAN_DRAGON_FILE = os.path.join(DATA_DIR, "asian_dragon_small.obj")

results_list = []
trained_models = {}

if not os.path.exists(TARGET_FILE):
  print(f"BŁĄD: Nie znaleziono pliku docelowego (teapot): {TARGET_FILE}")
  exit()

for name, path in OBJECTS.items():
  if not os.path.exists(path):
      print(f"Pominięto {name} - brak pliku: {path}")
      continue

  model, s_mesh, t_mesh = train_deformation(name, path, TARGET_FILE, epochs=1100)

  if model:
      trained_models[name] = model
      metrics = save_and_evaluate(model, s_mesh, t_mesh, name)
      results_list.append(metrics)
  del s_mesh
  del t_mesh
  torch.cuda.empty_cache()
  gc.collect()
  print(f"🧹 Pamięć wyczyszczona po {name}")
if "Dragon" in trained_models and os.path.exists(ASIAN_DRAGON_FILE):
  print("\n🧪 --- Eksperyment: Asian Dragon ---")
  print("Próba przekształcenia Asian Dragon za pomocą sieci nauczonej na Stanford Dragon.")


  asian_mesh = trimesh.load(ASIAN_DRAGON_FILE)
  canon = Canonicalizer()
  asian_mesh = canon.transform(asian_mesh)
  asian_mesh = canon.fit_transform(asian_mesh)

  dragon_model = trained_models["Dragon"]

  t_mesh = trimesh.load(TARGET_FILE)
  t_mesh = Canonicalizer().fit_transform(t_mesh)

  metrics = save_and_evaluate(dragon_model, asian_mesh, t_mesh, "Asian_Dragon_Test")
  results_list.append(metrics)

print("\n📋 --- PODSUMOWANIE WYNIKÓW ---")
df_results = pd.DataFrame(results_list)
print(df_results)

df_results.to_csv("./drive/MyDrive/sigk_4/results/metrics_summary.csv", index=False)


🚀 Rozpoczynam trening PRO: Bunny -> Teapot
   Epoka 0/1100 | Loss: 0.224518 | LR: 0.001000
   Epoka 100/1100 | Loss: 0.026640 | LR: 0.001000
   Epoka 200/1100 | Loss: 0.026177 | LR: 0.001000
   Epoka 300/1100 | Loss: 0.024741 | LR: 0.001000
   Epoka 400/1100 | Loss: 0.023568 | LR: 0.000500
   Epoka 500/1100 | Loss: 0.023483 | LR: 0.000500
   Epoka 600/1100 | Loss: 0.023281 | LR: 0.000250
   Epoka 700/1100 | Loss: 0.023045 | LR: 0.000125
   Epoka 800/1100 | Loss: 0.023078 | LR: 0.000063
   Epoka 900/1100 | Loss: 0.022846 | LR: 0.000063
   Epoka 1000/1100 | Loss: 0.022861 | LR: 0.000031
✅ Trening zakończony.
📊 Generowanie wyników dla: Bunny...
Zapisano ./drive/MyDrive/sigk_4/results/Bunny_step_0.obj
Zapisano ./drive/MyDrive/sigk_4/results/Bunny_step_50.obj
Zapisano ./drive/MyDrive/sigk_4/results/Bunny_step_100.obj
🧹 Pamięć wyczyszczona po Bunny

🚀 Rozpoczynam trening PRO: Dragon -> Teapot
   Epoka 0/1100 | Loss: 0.274958 | LR: 0.001000
   Epoka 100/1100 | Loss: 0.029878 | LR: 0.001000
 

AttributeError: 'TrackedArray' object has no attribute 'vertices'

In [10]:
torch.cuda.empty_cache()
gc.collect()

8